# பாடம் 08 - பன்முகப்படையாளர் வடிவமைப்பு மாதிரி


## அமைப்பு


In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

%pip install agent-framework azure-ai-projects azure-identity --quiet

import os
import asyncio

from agent_framework import AgentResponseUpdate, WorkflowBuilder
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## பன்முகச் செயலாளர் அமைப்புகள் ஏன்?

சஞ்சார திட்டமிடல் போன்ற உண்மையான உலகப் பணிகள் பல்வேறு வகையான நிபுணத்துவங்களை envolver செய்கின்றன — போக்குவரத்து, உள்ளூரான அறிவு, பட்ஜெட்டிங் ועוד. ஒரு தனி செயலாளர் அனைத்தையும் கையாள முயற்சிப்பது விரைவாக சுமையடையக்கூடியதாக மாறும்.

பன்முகச் செயலாளர் அமைப்புகள் இதனை **துவக்கமாக்கல்** மூலம் தீர்க்கின்றன: ஒவ்வொரு செயலாளரும் அவர்களது நிபுணத்துவப் புலத்தில் கவனம் செலுத்தி, பொதுவான செயலாளரைவிட உயர் தரமான முடிவுகளை உண்டாக்குகின்றனர். அவை **மேம்பட்ட அளவீடு**யையும் மேம்படுத்துகின்றன — புதிய செயலாளர்களை (உதா., விமான நிபுணர், உணவக விமர்சகர்) இல்லாத வேலைப்பயிற்சியை மறுபடியும் எழுதாமல் சேர்க்கலாம். செயலாளர்கள் ஒருவரது சூழலை அடுத்தவருக்கு அனுப்பும் கட்டமைப்பு வாயிலாக இணைக்கப்படுகின்றன.


## சிறப்பு முகவர்கள் உருவாக்கல்


In [ ]:
planner_agent = await provider.create_agent(
    name="TravelPlanner",
    instructions="You are a travel planning specialist. Create detailed trip itineraries based on the traveler's preferences. Include daily schedules, must-see attractions, and logistical tips.",
)

concierge_agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="You are a travel concierge who reviews and enhances trip plans. Review the plan for completeness, add local insider tips, suggest restaurants, and identify potential issues. Provide your feedback in a constructive format.",
)

## தொடர்ச்சியான பணிமுறை உருவாக்குதல்

`WorkflowBuilder` மூலம் உங்களுக்கு முகவர்கள் ஒரு வழிசெலுத்தப்பட்ட வரைபடத்தில் இணைக்கலாம். இங்கே எளிய இரண்டு படி குழாய் உருவாக்குகிறோம்: **TravelPlanner** பயண திட்டத்தை உருவாக்குகிறது, பின்னர் **TravelConcierge** அதனை மதிப்பாய்வு செய்து மேம்படுத்துகிறது.


In [ ]:
workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .build()

last_author = None
events = workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## செயலில் மேலும் முகவர்களைச் சேர்க்குதல்

பன்முகவர் மாதிரியின் மிகப்பெரிய நன்மைகளில் ஒன்று அதை எவ்வளவு எளிதாக விரிவுப்படுத்த முடியும் என்பதே ஆகும். கீழே நாம் பயணியரின் பட்ஜெட்டுக்கு எதிராக திட்டத்தை சரிபார்க்கும், செலவுகள் வரம்பிற்கு மேலாக செல்லக்கூடிய உள்ளடக்கங்களை குறிக்கவும், பணம் சேமிக்கும் மாற்றுகளை பரிந்துரைக்கும் **BudgetReviewer** முகவரைச் சேர்க்கின்றோம். இப்போது செயல் மூன்று முகவர்களை தொடர்ச்சியாக இயக்குகிறது:

```
TravelPlanner → TravelConcierge → BudgetReviewer
```


In [ ]:
budget_agent = await provider.create_agent(
    name="BudgetReviewer",
    instructions="You are a budget-conscious travel advisor. Review the proposed trip plan and concierge enhancements against the traveler's stated budget. Estimate costs for flights, hotels, meals, and activities. Flag anything that risks exceeding the budget and suggest cost-saving alternatives while preserving the trip's quality.",
)

extended_workflow = WorkflowBuilder(start_executor=planner_agent) \
    .add_edge(planner_agent, concierge_agent) \
    .add_edge(concierge_agent, budget_agent) \
    .build()

last_author = None
events = extended_workflow.run("Plan a 5-day trip to Paris for a food-loving couple on a $3000 budget.", stream=True)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## சுருக்கம்

இந்த பாடத்தில் நீங்கள் எப்படி செய்யவேண்டுமென்று கற்றுக்கொண்டீர்கள்:

1. **சிறப்பு முகவர்கள் உருவாக்குதல்** — ஒவ்வொருவரும் ஒரு நுண்ணிய பங்கு (திட்டமிடுதல், வரவேற்பாளர், பட்ஜெட் மதிப்பாய்வு) கொண்டவர்கள்.
2. **`WorkflowBuilder` மற்றும் `add_edge` பயன்படுத்தி முகவர்களை தொடர் பணிசெயலாக்கத்தில் இணைத்தல்**.
3. **பல முகவர் குழாயிலிருந்து வெளியீட்டை நேரடியாக வாசித்தல்**, எந்த முகவர் பேசுகிறான் என்பதை கண்காணித்தல்.
4. **ஏற்கனவே உள்ள முகவர்களை மாற்றாமல் புதிய முகவர்களை சங்கிலியில் சேர்த்துத் பணிசெயலாக்கத்தை நீட்டித்தல்**.

பல முகவர் வடிவமைப்பு முறை ஒவ்வொரு முகவரும் எளிமையாக இருக்கும்படி வைத்திருக்கிறது, அதே சமயம் தனித்தனியான முகவர் ஒருவனும் செய்ய முடியாதவாறு செறிவான, முழுமையாக மதிப்பாய்வு செய்யப்பட்ட முடிவுகளை உருவாக்குகின்றது.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**அறிக்கை**:
இந்த ஆவணம் AI மொழி பெயர்ப்பு சேவை [Co-op Translator](https://github.com/Azure/co-op-translator) பயன்படுத்தி மொழிபெயர்க்கப்பட்டுள்ளது. நாம் துல்லியத்திற்காக முயலினாலும், தானாகத் தயாரிக்கப்பட்ட மொழிபெயர்ப்புகளில் தவறுகள் அல்லது துல்லியநீக்கங்கள் இருக்கலாம் என்பதை கவனத்தில் கொள்ளவும். மூல ஆவணம் அதன் சொந்த மொழியில் அதிகாரபூர்வமானதாக கருதப்பட வேண்டும். மிக முக்கியமான தகவலுக்கு, தொழில்முறை மனித மொழிபெயர்ப்பை பரிந்துரைக்கிறோம். இந்த மொழிபெயர்ப்பின் பயன்பாட்டினால் ஏற்படும் எந்தவொரு தவறான புரிதலும் அல்லது தவறான விளக்கங்களுக்கான பொறுப்பும் எங்களுக்கு இல்லை.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
